<a href="https://colab.research.google.com/github/Arfa-Tariq/AstroPlanner-AI/blob/main/notebooks/03_celestial_visibility.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 — Celestial Visibility Engine

Computes which deep-sky objects (NGC/IC catalog, 13,000+ objects) are
realistically observable from the user's location across the upcoming
7-night window.

Unlike the weather module, visibility is pure orbital mechanics — no
forecast horizon, all 7 nights get full data.

### Pipeline
1. **Magnitude filter** — drops objects too faint for the user's telescope aperture
2. **Declination filter** — drops objects that never reach a useful altitude (>30°) from this latitude
3. **Nightly sky computation** — for surviving candidates, computes altitude throughout the astronomical night window using Astroplan, filtering by altitude (>30°), moon separation (>30°), and astronomical darkness

### Inputs (from Drive)
- `current_request.json` — user profile (location, telescope specs)

### Outputs (to Drive)
- `weekly_visibility.json` — visible objects per night for all 7 days, each with peak altitude, rise/set times, moon separation, magnitude, and object type

### Note
This module is purely deterministic — no LLM involved. The output feeds
directly into the Recommendation Engine (notebook 04), which ranks and
filters these candidates further based on experience level, equipment,
and sky quality scores from the weather module.

In [1]:
!pip install astropy astroplan requests -q

import sys, os, json, requests
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from typing import Optional
from google.colab import drive

drive.mount('/content/drive')

!git clone https://github.com/Arfa-Tariq/AstroPlanner-AI.git 2>/dev/null || git -C /content/AstroPlanner-AI pull

sys.path.append('/content/AstroPlanner-AI/src')

DATA_DIR = '/content/drive/MyDrive/AstroPlanner/data'
os.makedirs(DATA_DIR, exist_ok=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 5.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Mounted at /content/drive


In [2]:
from astropy.coordinates import EarthLocation, SkyCoord, AltAz, get_body
from astropy.time import Time
import astropy.units as u
from astroplan import Observer, FixedTarget
from astroplan import (
    AltitudeConstraint,
    MoonSeparationConstraint,
    AtNightConstraint,
    is_observable
)

from models import WeeklyPlanRequest, UserProfile

In [5]:
from astropy.utils import iers
from astropy.utils.iers import conf as iers_conf
iers_conf.auto_download = False
iers_conf.auto_max_age = None

import warnings
warnings.filterwarnings('ignore', message='.*IERS.*')
warnings.filterwarnings('ignore', message='.*NonRotation.*')
warnings.filterwarnings('ignore', message='.*failed to download.*')
warnings.filterwarnings('ignore', message='.*Angular separation.*')

In [6]:
with open(f'{DATA_DIR}/current_request.json') as f:
    plan_request = WeeklyPlanRequest.model_validate_json(f.read())

print(f"Computing visibility for: {plan_request.user.name}")
print(f"Location: {plan_request.user.latitude}°N, {plan_request.user.longitude}°E")
print(f"Telescope aperture: {plan_request.user.telescope.aperture_mm}mm")

Computing visibility for: Arfa
Location: 33.223°N, 32.44°E
Telescope aperture: 64.0mm


In [7]:
def load_ngc_catalog() -> pd.DataFrame:
    """
    Downloads and loads the OpenNGC catalog from GitHub — a clean,
    maintained CSV of all NGC/IC objects with coordinates, magnitudes,
    object types, and sizes. Cached locally to Drive so it's only
    downloaded once, not on every run.
    """
    cache_path = f'{DATA_DIR}/ngc_catalog.csv'

    if not os.path.exists(cache_path):
        print("Downloading OpenNGC catalog (first time only)...")
        url = "https://raw.githubusercontent.com/mattiaverga/OpenNGC/master/database_files/NGC.csv"
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        with open(cache_path, 'w') as f:
            f.write(response.text)
        print(f"Saved to {cache_path}")
    else:
        print("Loaded NGC catalog from cache.")

    df = pd.read_csv(cache_path, sep=';', low_memory=False)

    # Keep only columns we actually use
    cols = ['Name', 'Type', 'RA', 'Dec', 'V-Mag', 'B-Mag',
            'MajAx', 'MinAx', 'Common names']
    df = df[[c for c in cols if c in df.columns]]

    # Prefer V-Mag, fall back to B-Mag for magnitude
    df['magnitude'] = pd.to_numeric(df['V-Mag'], errors='coerce')
    df.loc[df['magnitude'].isna(), 'magnitude'] = pd.to_numeric(
        df.loc[df['magnitude'].isna(), 'B-Mag'], errors='coerce'
    )

    print(f"Total objects in catalog: {len(df)}")
    return df

ngc_df = load_ngc_catalog()
ngc_df.head()

Loaded NGC catalog from cache.
Total objects in catalog: 13969


,Name,Type,RA,Dec,V-Mag,B-Mag,MajAx,MinAx,Common names,magnitude
0,IC0001,**,00:08:27.05,+27:43:03.6,NaN,NaN,NaN,NaN,NaN,NaN
1,IC0002,G,00:11:00.88,-12:49:22.3,NaN,15.46,0.98,0.32,NaN,15.46
2,IC0003,G,00:12:06.09,-00:24:54.8,NaN,14.78,0.93,0.67,NaN,14.78
3,IC0004,G,00:13:26.94,+17:29:11.2,NaN,14.14,1.17,0.84,NaN,14.14
4,IC0005,G,00:17:34.93,-09:32:36.1,NaN,14.57,0.99,0.66,NaN,14.57


In [8]:
def compute_limiting_magnitude(aperture_mm: float) -> float:
    """
    Estimates the visual limiting magnitude achievable with this telescope
    aperture using the standard formula: 2.1 + 5 * log10(aperture_mm).
    This is a conservative estimate for visual observing; imaging can go
    ~2 magnitudes deeper but we'll let the recommendation engine handle
    that distinction. Used here purely to pre-filter the catalog before
    any expensive sky calculations.
    """
    return 2.1 + 5 * np.log10(aperture_mm)


def prefilter_catalog(df: pd.DataFrame, user: UserProfile) -> pd.DataFrame:
    """
    Stage 1 & 2 of the visibility pipeline: cheap filtering that eliminates
    the majority of catalog objects without any sky position computation.

    Stage 1 — Magnitude filter: drops objects fainter than the telescope's
    limiting magnitude. Objects with no magnitude data are kept (they may
    still be observable, e.g. some bright emission nebulae have unreliable
    catalog magnitudes).

    Stage 2 — Declination filter: drops objects that can never reach a
    useful altitude (>= 30°) from this latitude. At latitude φ, an object
    at declination δ has maximum altitude = 90° - |φ - δ|. Objects where
    this is < 30° are never usefully observable from this site, regardless
    of the night.
    """
    lat = user.latitude
    limiting_mag = compute_limiting_magnitude(user.telescope.aperture_mm)
    print(f"Telescope limiting magnitude: {limiting_mag:.1f}")

    # Parse coordinates — OpenNGC stores RA as HH:MM:SS.ss and Dec as ±DD:MM:SS.ss
    def parse_ra(ra_str):
        try:
            parts = str(ra_str).split(':')
            return float(parts[0]) * 15 + float(parts[1]) * 0.25 + float(parts[2]) * (15/3600)
        except:
            return np.nan

    def parse_dec(dec_str):
        try:
            parts = str(dec_str).split(':')
            sign = -1 if str(dec_str).startswith('-') else 1
            return sign * (abs(float(parts[0])) + float(parts[1])/60 + float(parts[2])/3600)
        except:
            return np.nan

    df = df.copy()
    df['ra_deg'] = df['RA'].apply(parse_ra)
    df['dec_deg'] = df['Dec'].apply(parse_dec)

    # Drop objects with unparseable coordinates
    df = df.dropna(subset=['ra_deg', 'dec_deg'])

    before = len(df)

    # Stage 1: magnitude filter
    df = df[
        df['magnitude'].isna() |
        (df['magnitude'] <= limiting_mag)
    ]
    print(f"After magnitude filter ({limiting_mag:.1f}): {len(df)} objects (removed {before - len(df)})")

    # Stage 2: declination / max altitude filter
    before = len(df)
    df['max_altitude'] = 90 - abs(lat - df['dec_deg'])
    df = df[df['max_altitude'] >= 30]
    print(f"After declination filter (lat={lat}°): {len(df)} objects (removed {before - len(df)})")

    return df.reset_index(drop=True)


filtered_df = prefilter_catalog(ngc_df, plan_request.user)
print(f"\nCandidates for sky computation: {len(filtered_df)}")

Telescope limiting magnitude: 11.1
After magnitude filter (11.1): 3482 objects (removed 10480)
After declination filter (lat=33.223°): 2777 objects (removed 705)

Candidates for sky computation: 2777


In [20]:
def get_night_window(observer: Observer, date_str: str):
    """
    Returns (night_start, night_end) as Astropy Time objects representing
    astronomical twilight for the given date at this observer's location.
    """
    midnight = Time(f"{date_str} 23:59:00")
    night_start = observer.twilight_evening_astronomical(midnight, which='nearest')
    night_end = observer.twilight_morning_astronomical(midnight, which='nearest')
    return night_start, night_end


def compute_nightly_visibility(
    filtered_df: pd.DataFrame,
    user: UserProfile,
    date_str: str
) -> list[dict]:
    """
    Stage 3 of the visibility pipeline. Two key performance decisions:
    1. AtNightConstraint removed — redundant since time_range is already
       bounded to the astronomical night window, and it was triggering
       expensive GCRS frame transformations internally causing both the
       warning spam and the slowdown.
    2. Rise/set times derived from the altitude grid we already compute
       rather than calling observer.target_rise_time() per object, which
       was making 2 × N Astroplan calls for every visible object.
    """
    observer = Observer(
        location=EarthLocation(
            lat=user.latitude * u.deg,
            lon=user.longitude * u.deg,
            height=0 * u.m
        ),
        name="observer"
    )

    night_start, night_end = get_night_window(observer, date_str)
    duration_hours = (night_end - night_start).to(u.hour).value

    if duration_hours <= 0:
        return []

    midnight = Time(f"{date_str} 23:59:00")
    moon = get_body('moon', midnight, ephemeris='builtin')
    moon_coord = SkyCoord(
        ra=moon.ra.deg * u.deg,
        dec=moon.dec.deg * u.deg,
        frame='icrs'
    )

    # AltitudeConstraint + MoonSeparationConstraint only — no AtNightConstraint.
    # We're already passing time_range=[night_start, night_end] to is_observable,
    # so night-only is guaranteed without needing the constraint.
    constraints = [
        AltitudeConstraint(min=30 * u.deg),
        MoonSeparationConstraint(min=30 * u.deg),
    ]

    # Build all targets at once for vectorized check
    targets = []
    valid_rows = []
    for _, row in filtered_df.iterrows():
        try:
            coord = SkyCoord(ra=row['ra_deg'] * u.deg, dec=row['dec_deg'] * u.deg)
            targets.append(FixedTarget(coord=coord, name=str(row['Name'])))
            valid_rows.append(row)
        except Exception:
            continue

    if not targets:
        return []

    # Single vectorized observability check
    observable_mask = is_observable(
        constraints, observer, targets,
        time_range=[night_start, night_end]
    )

    # Build time grid for altitude computation
    n_steps = max(int(duration_hours * 2), 2)
    time_grid = night_start + np.linspace(0, duration_hours, n_steps) * u.hour
    time_labels = [
        (night_start + timedelta(hours=i * duration_hours / (n_steps - 1)))
        .to_datetime().strftime('%H:%M')
        for i in range(n_steps)
    ]

    visible = []
    for target, row, is_obs in zip(targets, valid_rows, observable_mask):
        if not is_obs:
            continue
        try:
            coord = target.coord
            altaz_frame = AltAz(obstime=time_grid, location=observer.location)
            altaz = coord.transform_to(altaz_frame)
            alts = altaz.alt.deg

            peak_idx = int(np.argmax(alts))
            peak_alt = float(alts[peak_idx])
            peak_time = time_labels[peak_idx]

            # Derive rise/set from altitude grid crossings (no extra Astroplan calls)
            above_horizon = alts >= 30
            rising_indices = np.where(above_horizon)[0]
            rise_time = time_labels[rising_indices[0]] if len(rising_indices) > 0 else "circumpolar"
            set_time = time_labels[rising_indices[-1]] if len(rising_indices) > 0 else "circumpolar"

            moon_sep = float(moon_coord.separation(coord).deg)

            visible.append({
                "name": str(row['Name']),
                "common_name": str(row.get('Common names', '')) or None,
                "type": str(row.get('Type', 'Unknown')),
                "magnitude": float(row['magnitude']) if not pd.isna(row['magnitude']) else None,
                "size_arcmin": float(row['MajAx']) if 'MajAx' in row and not pd.isna(row.get('MajAx')) else None,
                "ra_deg": round(row['ra_deg'], 4),
                "dec_deg": round(row['dec_deg'], 4),
                "peak_altitude_deg": round(peak_alt, 1),
                "peak_time_local": peak_time,
                "rise_time_local": rise_time,
                "set_time_local": set_time,
                "moon_separation_deg": round(moon_sep, 1),
            })

        except Exception:
            continue
    # Sort by peak altitude, cap at top 100 per night — the recommendation
    # engine (notebook 04) will filter this down further by experience level,
    # equipment match, and sky quality. 100 is enough headroom for it to work
    # with while keeping the JSON a sane size.
    visible.sort(key=lambda x: x['peak_altitude_deg'], reverse=True)
    return visible[:100]

In [21]:
def get_weekly_visibility(user: UserProfile, filtered_df: pd.DataFrame) -> list[dict]:
    """
    Computes visible celestial objects for all 7 nights of the observation
    window. Unlike the weather module, visibility has no forecast horizon —
    orbital mechanics are computable for any date — so all 7 nights get
    full data. This is the function that becomes a LangChain tool in
    notebook 06.
    """
    today = datetime.today()
    weekly_visibility = []

    for offset in range(7):
        date_str = (today + timedelta(days=offset)).strftime('%Y-%m-%d')
        print(f"Computing visibility for {date_str}...", end=" ")

        nightly = compute_nightly_visibility(filtered_df, user, date_str)
        print(f"{len(nightly)} objects visible")

        weekly_visibility.append({
            "date": date_str,
            "day_offset": offset,
            "visible_object_count": len(nightly),
            "objects": nightly,
        })

    return weekly_visibility

In [22]:
weekly_visibility = get_weekly_visibility(plan_request.user, filtered_df)

with open(f'{DATA_DIR}/weekly_visibility.json', 'w') as f:
    json.dump(weekly_visibility, f, indent=2, default=str)

print(f"\nSaved to {DATA_DIR}/weekly_visibility.json")

# Quick summary per night
print("\n=== Visibility Summary ===")
for night in weekly_visibility:
    print(f"{night['date']}: {night['visible_object_count']} objects visible")

Computing visibility for 2026-07-01... 100 objects visible
Computing visibility for 2026-07-02... 100 objects visible
Computing visibility for 2026-07-03... 100 objects visible
Computing visibility for 2026-07-04... 100 objects visible
Computing visibility for 2026-07-05... 100 objects visible
Computing visibility for 2026-07-06... 100 objects visible
Computing visibility for 2026-07-07... 100 objects visible

Saved to /content/drive/MyDrive/AstroPlanner/data/weekly_visibility.json

=== Visibility Summary ===
2026-07-01: 100 objects visible
2026-07-02: 100 objects visible
2026-07-03: 100 objects visible
2026-07-04: 100 objects visible
2026-07-05: 100 objects visible
2026-07-06: 100 objects visible
2026-07-07: 100 objects visible


In [23]:
# Find the night with the most visible objects
best_night = max(weekly_visibility, key=lambda n: n['visible_object_count'])
print(f"Best night: {best_night['date']} ({best_night['visible_object_count']} objects)\n")
print("Top 10 objects by altitude:")
for obj in best_night['objects'][:10]:
    print(
        f"  {obj['name']:10} {obj['common_name'] or '':20} "
        f"type={obj['type']:4} mag={obj['magnitude']} "
        f"peak={obj['peak_altitude_deg']}° at {obj['peak_time_local']} "
        f"moon_sep={obj['moon_separation_deg']}°"
    )

Best night: 2026-07-01 (100 objects)

Top 10 objects by altitude:
  NGC6089    nan                  type=GPair mag=None peak=89.0° at 19:19 moon_sep=80.5°
  NGC6720    Ring Nebula          type=PN   mag=8.8 peak=88.6° at 22:11 moon_sep=58.8°
  NGC7037    nan                  type=OCl  mag=None peak=88.5° at 00:28 moon_sep=57.2°
  IC1310     nan                  type=Cl+N mag=None peak=88.2° at 23:19 moon_sep=57.1°
  NGC6666    nan                  type=Other mag=None peak=88.1° at 21:37 moon_sep=61.0°
  NGC6979    nan                  type=SNR  mag=None peak=88.1° at 23:54 moon_sep=54.7°
  NGC6974    nan                  type=SNR  mag=None peak=87.9° at 23:54 moon_sep=54.5°
  NGC6960    Veil Nebula,Filamentary Nebula,Western Veil type=SNR  mag=7.0 peak=87.4° at 23:54 moon_sep=53.1°
  NGC6871    nan                  type=OCl  mag=5.2 peak=87.3° at 23:19 moon_sep=58.0°
  NGC6883    nan                  type=OCl  mag=8.0 peak=87.3° at 23:19 moon_sep=58.0°
